In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "src" / "feature_engineering.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найден корень проекта. Current directory: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_raw = train.drop(columns=[TARGET_COLUMN]).copy()
y = train[TARGET_COLUMN].copy()

X_features = add_title_hierarchy_features(
    prepare_features(X_raw)
)

price_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

train_idx, valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.20,
    random_state=42,
    stratify=price_bins,
)


def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


print("Train rows:", len(train_idx))
print("Valid rows:", len(valid_idx))
print("Feature matrix:", X_features.shape)

Train rows: 6672
Valid rows: 1668
Feature matrix: (8340, 66)


Три набора признаков

In [2]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

EXCLUDED_COLUMNS_V6 = EXCLUDED_COLUMNS + [
    "Полное название",
]

lean_core_columns = [
    "Бренд",
    "Год выпуска",
    "Модель",
    "Тип машины",
    "Исползование",
    "КПП",
    "Привод",
    "Топливо",
    "Тип кузова",
    "Оценка эксперта",
    "Количество владельцев",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
    "Название_префикс_2",
    "Название_префикс_3",
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_моторный_маркер",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

lean_market_extra_columns = [
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
    "Штат",
]

full_v6_columns = [
    column
    for column in X_features.columns
    if column not in EXCLUDED_COLUMNS_V6
]

lean_market_columns = list(
    dict.fromkeys(
        lean_core_columns
        + lean_market_extra_columns
    )
)

feature_sets = {
    "full_v6": full_v6_columns,
    "lean_core": lean_core_columns,
    "lean_market": lean_market_columns,
}

for feature_set_name, columns in feature_sets.items():
    missing_columns = [
        column
        for column in columns
        if column not in X_features.columns
    ]

    assert not missing_columns, (
        f"{feature_set_name}: отсутствуют признаки {missing_columns}"
    )

    assert "Полное название" not in columns

    print(
        f"{feature_set_name}: "
        f"{len(columns)} features"
    )

full_v6: 58 features
lean_core: 44 features
lean_market: 51 features


Подготовка CatBoost-матриц

In [3]:
def prepare_catboost_matrix(
    features: pd.DataFrame,
    feature_columns: list[str],
):
    result = features[
        feature_columns
    ].copy()

    numeric_columns = result.select_dtypes(
        include=["number", "bool"],
    ).columns.tolist()

    categorical_columns = [
        column
        for column in feature_columns
        if column not in numeric_columns
    ]

    for column in categorical_columns:
        result[column] = (
            result[column]
            .fillna("__MISSING__")
            .astype(str)
        )

    return result, categorical_columns


prepared_feature_sets = {}

for feature_set_name, columns in feature_sets.items():
    X_model, categorical_columns = prepare_catboost_matrix(
        X_features,
        columns,
    )

    prepared_feature_sets[feature_set_name] = {
        "X": X_model,
        "categorical_columns": categorical_columns,
    }

    print(
        f"{feature_set_name}: "
        f"{X_model.shape}, "
        f"categorical={len(categorical_columns)}"
    )

full_v6: (8340, 58), categorical=18
lean_core: (8340, 44), categorical=11
lean_market: (8340, 51), categorical=12


4. Отдельно создаём test-like subset внутри holdout

Это важнее общего holdout MAPE.

Мы проверим качество на validation-объектах, чьё сырое Полное название не встречалось в train-части holdout. Такой subset ближе к настоящему hidden test.

In [4]:
def normalize_raw_title(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .fillna("__MISSING__")
        .str.upper()
        .str.strip()
    )


title_train_part = normalize_raw_title(
    X_raw.iloc[train_idx]["Полное название"]
)

title_valid_part = normalize_raw_title(
    X_raw.iloc[valid_idx]["Полное название"]
)

seen_titles = set(title_train_part)

valid_title_unseen_mask = (
    ~title_valid_part.isin(seen_titles)
).to_numpy()

y_valid = y.iloc[valid_idx].to_numpy()
low_price_threshold = y.iloc[train_idx].quantile(0.25)

valid_low_price_mask = (
    y_valid <= low_price_threshold
)

print(
    "Validation rows with unseen raw title:",
    valid_title_unseen_mask.sum(),
)

print(
    "Share of unseen raw titles:",
    f"{valid_title_unseen_mask.mean() * 100:.2f}%",
)

print(
    "Low-price validation rows:",
    valid_low_price_mask.sum(),
)

Validation rows with unseen raw title: 865
Share of unseen raw titles: 51.86%
Low-price validation rows: 418


5. Быстрый прогон моделей

In [5]:
model_configs = {
    "cb_depth8_l2_5": {
        "depth": 8,
        "l2_leaf_reg": 5,
    },
    "cb_depth6_l2_10": {
        "depth": 6,
        "l2_leaf_reg": 10,
    },
}

experiment_rows = []
holdout_predictions = {}

for feature_set_name, prepared_data in prepared_feature_sets.items():
    X_model = prepared_data["X"]
    categorical_columns = prepared_data[
        "categorical_columns"
    ]

    X_train_fold = X_model.iloc[train_idx].copy()
    X_valid_fold = X_model.iloc[valid_idx].copy()

    for config_name, config in model_configs.items():
        print(
            f"\nRunning: {feature_set_name} | {config_name}"
        )

        model = CatBoostRegressor(
            loss_function="RMSE",
            iterations=3000,
            learning_rate=0.05,
            depth=config["depth"],
            l2_leaf_reg=config["l2_leaf_reg"],
            random_seed=42,
            verbose=500,
            allow_writing_files=False,
        )

        model.fit(
            X_train_fold,
            np.log1p(y.iloc[train_idx]),
            cat_features=categorical_columns,
            eval_set=(
                X_valid_fold,
                np.log1p(y.iloc[valid_idx]),
            ),
            use_best_model=True,
            early_stopping_rounds=200,
        )

        valid_pred = np.maximum(
            np.expm1(
                model.predict(X_valid_fold)
            ),
            1,
        )

        experiment_name = (
            f"{feature_set_name}__{config_name}"
        )

        holdout_predictions[
            experiment_name
        ] = valid_pred

        experiment_rows.append(
            {
                "experiment": experiment_name,
                "feature_set": feature_set_name,
                "model_config": config_name,
                "n_features": X_model.shape[1],
                "best_iteration": model.get_best_iteration(),
                "overall_mape_pct": mape_percent(
                    y_valid,
                    valid_pred,
                ),
                "unseen_title_mape_pct": mape_percent(
                    y_valid[valid_title_unseen_mask],
                    valid_pred[valid_title_unseen_mask],
                ),
                "low_price_mape_pct": mape_percent(
                    y_valid[valid_low_price_mask],
                    valid_pred[valid_low_price_mask],
                ),
            }
        )

experiment_results = (
    pd.DataFrame(experiment_rows)
    .sort_values("overall_mape_pct")
    .reset_index(drop=True)
)

display(experiment_results)


Running: full_v6 | cb_depth8_l2_5
0:	learn: 0.6538509	test: 0.6509994	best: 0.6509994 (0)	total: 263ms	remaining: 13m 7s
500:	learn: 0.1568077	test: 0.2042800	best: 0.2042780 (499)	total: 1m 1s	remaining: 5m 8s
1000:	learn: 0.1200134	test: 0.1934439	best: 0.1934330 (994)	total: 2m 29s	remaining: 4m 58s
1500:	learn: 0.0982768	test: 0.1898236	best: 0.1897616 (1489)	total: 4m 1s	remaining: 4m 1s
2000:	learn: 0.0823696	test: 0.1882054	best: 0.1881731 (1994)	total: 5m 51s	remaining: 2m 55s
2500:	learn: 0.0693404	test: 0.1871944	best: 0.1871944 (2500)	total: 7m 37s	remaining: 1m 31s
2999:	learn: 0.0585626	test: 0.1864078	best: 0.1864078 (2999)	total: 9m 28s	remaining: 0us

bestTest = 0.1864077782
bestIteration = 2999


Running: full_v6 | cb_depth6_l2_10
0:	learn: 0.6554337	test: 0.6527769	best: 0.6527769 (0)	total: 122ms	remaining: 6m 5s
500:	learn: 0.2003011	test: 0.2168220	best: 0.2168220 (500)	total: 1m 1s	remaining: 5m 5s
1000:	learn: 0.1673878	test: 0.2011490	best: 0.2011463 (999)	tota

,experiment,feature_set,model_config,n_features,best_iteration,overall_mape_pct,unseen_title_mape_pct,low_price_mape_pct
0,full_v6__cb_depth8_l2_5,full_v6,cb_depth8_l2_5,58,2999,12.908705,16.565257,18.724338
1,full_v6__cb_depth6_l2_10,full_v6,cb_depth6_l2_10,58,2999,13.095100,16.643747,18.322036
2,lean_market__cb_depth8_l2_5,lean_market,cb_depth8_l2_5,51,2998,13.516475,17.435375,19.835939
3,lean_market__cb_depth6_l2_10,lean_market,cb_depth6_l2_10,51,2998,13.519077,17.268634,19.472209
4,lean_core__cb_depth8_l2_5,lean_core,cb_depth8_l2_5,44,2794,18.887041,23.983425,29.643569
5,lean_core__cb_depth6_l2_10,lean_core,cb_depth6_l2_10,44,2987,19.314223,24.494436,29.511008


Гипотеза про «компактная модель лучше переносится» не подтвердилась.

FULL_V6:      12.909% overall | 16.565% unseen
LEAN_MARKET:  13.516% overall | 17.435% unseen
LEAN_CORE:    18.887% overall | 23.983% unseen
Вывод
LEAN_CORE — сразу в архив. Без пробега, двигателя и прочих рыночных признаков цена практически не восстанавливается.
LEAN_MARKET тоже хуже full v6 на всех трёх срезах:
+0.61 п.п. overall;
+0.87 п.п. на unseen title;
+1.11 п.п. на дешёвых машинах.
Более регуляризованный CatBoost (depth=6, l2=10) не решил проблему. Он чуть лучше на дешёвых машинах, но проиграл по общему MAPE.

То есть не «лишние шумные фичи» вредят current v6. Наоборот: семь исключённых признаков несут сильный полезный сигнал.

Теперь не надо считать OOF для lean-вариантов. Но эксперимент очень полезен: он сузил поиск.

Следующий шаг — определить, какие именно 7 признаков дали эту разницу. Выполни в этом же ноутбуке:

In [6]:
removed_from_full_v6 = [
    column
    for column in full_v6_columns
    if column not in lean_market_columns
]

print(
    "Признаки, которые есть в FULL_V6, "
    "но отсутствуют в LEAN_MARKET:"
)

for i, column in enumerate(removed_from_full_v6, start=1):
    print(f"{i}. {column}")

print("\nКоличество:", len(removed_from_full_v6))

Признаки, которые есть в FULL_V6, но отсутствуют в LEAN_MARKET:
1. Двигатель
2. Цвет
3. Локация
4. Название_без_года
5. Название_число_слов
6. Название_нормализованное_без_года
7. Название_префикс_4

Количество: 7


Шаг 1. Разделяем семь признаков на две смысловые группы

In [7]:
raw_context_features = [
    "Двигатель",
    "Цвет",
    "Локация",
]

title_detail_features = [
    "Название_без_года",
    "Название_число_слов",
    "Название_нормализованное_без_года",
    "Название_префикс_4",
]

Шаг 2. Быстрый групповой ablation на том же holdout

Вставь в текущий ноутбук после предыдущих экспериментов.

In [8]:
id="x9j6sk"
raw_context_features = [
    "Двигатель",
    "Цвет",
    "Локация",
]

title_detail_features = [
    "Название_без_года",
    "Название_число_слов",
    "Название_нормализованное_без_года",
    "Название_префикс_4",
]

feature_group_experiments = {
    "full_v6": full_v6_columns,
    "full_minus_raw_context": [
        column
        for column in full_v6_columns
        if column not in raw_context_features
    ],
    "full_minus_title_detail": [
        column
        for column in full_v6_columns
        if column not in title_detail_features
    ],
}

for name, columns in feature_group_experiments.items():
    print(name, len(columns))

full_v6 58
full_minus_raw_context 55
full_minus_title_detail 54


In [9]:
id="poo2xk"
def run_catboost_holdout(
    feature_columns,
    iterations=1500,
    depth=8,
    l2_leaf_reg=5,
):
    X_model, categorical_columns = prepare_catboost_matrix(
        X_features,
        feature_columns,
    )

    X_train_fold = X_model.iloc[train_idx].copy()
    X_valid_fold = X_model.iloc[valid_idx].copy()

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=iterations,
        learning_rate=0.05,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model.fit(
        X_train_fold,
        np.log1p(y.iloc[train_idx]),
        cat_features=categorical_columns,
        eval_set=(
            X_valid_fold,
            np.log1p(y.iloc[valid_idx]),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    valid_pred = np.maximum(
        np.expm1(model.predict(X_valid_fold)),
        1,
    )

    return model, valid_pred

In [10]:
id="qc3g61"
group_rows = []
group_predictions = {}

for experiment_name, feature_columns in feature_group_experiments.items():
    print(f"\nRunning: {experiment_name}")

    model, valid_pred = run_catboost_holdout(
        feature_columns=feature_columns,
        iterations=1500,
        depth=8,
        l2_leaf_reg=5,
    )

    group_predictions[experiment_name] = valid_pred

    group_rows.append(
        {
            "experiment": experiment_name,
            "n_features": len(feature_columns),
            "best_iteration": model.get_best_iteration(),
            "overall_mape_pct": mape_percent(
                y_valid,
                valid_pred,
            ),
            "unseen_title_mape_pct": mape_percent(
                y_valid[valid_title_unseen_mask],
                valid_pred[valid_title_unseen_mask],
            ),
            "low_price_mape_pct": mape_percent(
                y_valid[valid_low_price_mask],
                valid_pred[valid_low_price_mask],
            ),
        }
    )

group_results = (
    pd.DataFrame(group_rows)
    .sort_values("overall_mape_pct")
    .reset_index(drop=True)
)

display(group_results)


Running: full_v6
0:	learn: 0.6538509	test: 0.6509994	best: 0.6509994 (0)	total: 69.7ms	remaining: 1m 44s
500:	learn: 0.1568077	test: 0.2042800	best: 0.2042780 (499)	total: 41.2s	remaining: 1m 22s
1000:	learn: 0.1200134	test: 0.1934439	best: 0.1934330 (994)	total: 1m 20s	remaining: 40s
1499:	learn: 0.0982772	test: 0.1898236	best: 0.1897616 (1489)	total: 2m 26s	remaining: 0us

bestTest = 0.1897615961
bestIteration = 1489

Shrink model to first 1490 iterations.

Running: full_minus_raw_context
0:	learn: 0.6529284	test: 0.6510963	best: 0.6510963 (0)	total: 96.8ms	remaining: 2m 25s
500:	learn: 0.1571767	test: 0.2060989	best: 0.2060989 (500)	total: 47.1s	remaining: 1m 33s
1000:	learn: 0.1227710	test: 0.1968848	best: 0.1968787 (998)	total: 1m 40s	remaining: 50.3s
1499:	learn: 0.1021583	test: 0.1935605	best: 0.1935441 (1492)	total: 2m 28s	remaining: 0us

bestTest = 0.1935440672
bestIteration = 1492

Shrink model to first 1493 iterations.

Running: full_minus_title_detail
0:	learn: 0.6522418	t

,experiment,n_features,best_iteration,overall_mape_pct,unseen_title_mape_pct,low_price_mape_pct
0,full_v6,58,1489,13.234677,16.884548,19.263660
1,full_minus_title_detail,54,1499,13.256054,16.884182,18.868521
2,full_minus_raw_context,55,1492,13.480780,17.411255,20.019043


Прогоним четыре варианта на том же holdout и уже с iterations=3000:

In [11]:
single_feature_ablation = {
    "full_v6": full_v6_columns,
    "full_minus_engine": [
        column
        for column in full_v6_columns
        if column != "Двигатель"
    ],
    "full_minus_location": [
        column
        for column in full_v6_columns
        if column != "Локация"
    ],
    "full_minus_color": [
        column
        for column in full_v6_columns
        if column != "Цвет"
    ],
}

for name, columns in single_feature_ablation.items():
    print(
        f"{name}: {len(columns)} features | "
        f"engine={'Двигатель' in columns} | "
        f"location={'Локация' in columns} | "
        f"color={'Цвет' in columns}"
    )

full_v6: 58 features | engine=True | location=True | color=True
full_minus_engine: 57 features | engine=False | location=True | color=True
full_minus_location: 57 features | engine=True | location=False | color=True
full_minus_color: 57 features | engine=True | location=True | color=False


Теперь сам прогон. Здесь iterations=3000, остальная конфигурация полностью совпадает с текущим сильным full_v6.

In [12]:
single_ablation_rows = []
single_ablation_predictions = {}

for experiment_name, feature_columns in single_feature_ablation.items():
    print(f"\n{'=' * 70}")
    print(f"Running: {experiment_name}")
    print(f"{'=' * 70}")

    model, valid_pred = run_catboost_holdout(
        feature_columns=feature_columns,
        iterations=3000,
        depth=8,
        l2_leaf_reg=5,
    )

    single_ablation_predictions[experiment_name] = valid_pred

    single_ablation_rows.append(
        {
            "experiment": experiment_name,
            "n_features": len(feature_columns),
            "best_iteration": model.get_best_iteration(),
            "overall_mape_pct": mape_percent(
                y_valid,
                valid_pred,
            ),
            "unseen_title_mape_pct": mape_percent(
                y_valid[valid_title_unseen_mask],
                valid_pred[valid_title_unseen_mask],
            ),
            "low_price_mape_pct": mape_percent(
                y_valid[valid_low_price_mask],
                valid_pred[valid_low_price_mask],
            ),
        }
    )

single_ablation_results = (
    pd.DataFrame(single_ablation_rows)
    .sort_values("overall_mape_pct")
    .reset_index(drop=True)
)

display(single_ablation_results)


Running: full_v6
0:	learn: 0.6538509	test: 0.6509994	best: 0.6509994 (0)	total: 81ms	remaining: 4m 2s
500:	learn: 0.1568077	test: 0.2042800	best: 0.2042780 (499)	total: 59s	remaining: 4m 54s
1000:	learn: 0.1200134	test: 0.1934439	best: 0.1934330 (994)	total: 2m 10s	remaining: 4m 19s
1500:	learn: 0.0982768	test: 0.1898236	best: 0.1897616 (1489)	total: 3m 26s	remaining: 3m 26s
2000:	learn: 0.0823696	test: 0.1882054	best: 0.1881731 (1994)	total: 4m 42s	remaining: 2m 21s
2500:	learn: 0.0693404	test: 0.1871944	best: 0.1871944 (2500)	total: 5m 51s	remaining: 1m 10s
2999:	learn: 0.0585626	test: 0.1864078	best: 0.1864078 (2999)	total: 6m 58s	remaining: 0us

bestTest = 0.1864077782
bestIteration = 2999


Running: full_minus_engine
0:	learn: 0.6525012	test: 0.6505381	best: 0.6505381 (0)	total: 117ms	remaining: 5m 52s
500:	learn: 0.1558870	test: 0.2039983	best: 0.2039983 (500)	total: 54.8s	remaining: 4m 33s
1000:	learn: 0.1216303	test: 0.1952717	best: 0.1952717 (1000)	total: 2m 5s	remaining: 4m 

,experiment,n_features,best_iteration,overall_mape_pct,unseen_title_mape_pct,low_price_mape_pct
0,full_minus_color,57,2968,12.806429,16.547210,18.769882
1,full_v6,58,2999,12.908705,16.565257,18.724338
2,full_minus_engine,57,2994,12.959822,16.719902,18.593154
3,full_minus_location,57,2998,13.273089,17.208265,19.805294


In [13]:
full_row = single_ablation_results.loc[
    single_ablation_results["experiment"] == "full_v6"
].iloc[0]

single_ablation_comparison = single_ablation_results.copy()

for metric in [
    "overall_mape_pct",
    "unseen_title_mape_pct",
    "low_price_mape_pct",
]:
    single_ablation_comparison[
        f"delta_vs_full_{metric}"
    ] = (
        single_ablation_comparison[metric]
        - full_row[metric]
    )

display(
    single_ablation_comparison[
        [
            "experiment",
            "n_features",
            "best_iteration",
            "overall_mape_pct",
            "delta_vs_full_overall_mape_pct",
            "unseen_title_mape_pct",
            "delta_vs_full_unseen_title_mape_pct",
            "low_price_mape_pct",
            "delta_vs_full_low_price_mape_pct",
        ]
    ]
)

,experiment,n_features,best_iteration,overall_mape_pct,delta_vs_full_overall_mape_pct,unseen_title_mape_pct,delta_vs_full_unseen_title_mape_pct,low_price_mape_pct,delta_vs_full_low_price_mape_pct
0,full_minus_color,57,2968,12.806429,-0.102276,16.547210,-0.018047,18.769882,0.045544
1,full_v6,58,2999,12.908705,0.000000,16.565257,0.000000,18.724338,0.000000
2,full_minus_engine,57,2994,12.959822,0.051117,16.719902,0.154645,18.593154,-0.131184
3,full_minus_location,57,2998,13.273089,0.364384,17.208265,0.643008,19.805294,1.080956


Что видно надёжнее:

Без Локации: 13.273%  → Локация очень важна.
Без Двигателя: 12.960% → Двигатель полезен, особенно на unseen title.
Без Цвета: 12.806%    → Цвет может добавлять шум / переобучение.

Цвет — типичный кандидат на шумный признак: может слегка коррелировать с ценой внутри train, но не иметь устойчивой рыночной причинности. Это как раз могло ухудшать перенос на hidden test.

Ещё важный сигнал: best_iteration у полной v6 — 2999, то есть модель упёрлась в лимит 3000. Позже стоит проверить 4000–5000 итераций, но сначала подтвердим эффект удаления цвета.

Следующий шаг: repeated holdout для v6 и v7

Мы возьмём три разных stratified holdout-разбиения и в каждом обучим:

v6 = Full V6
v7 = Full V6 без Цвета

Если v7 выигрывает в среднем хотя бы на 0.07–0.10 п.п. и не проваливается на большинстве разбиений — считаем OOF и проверяем её вклад в ансамбль.

In [14]:
from sklearn.model_selection import train_test_split


feature_sets_repeated = {
    "full_v6": full_v6_columns,
    "full_minus_color": [
        column
        for column in full_v6_columns
        if column != "Цвет"
    ],
}

prepared_repeated_sets = {}

for experiment_name, feature_columns in feature_sets_repeated.items():
    X_model, categorical_columns = prepare_catboost_matrix(
        X_features,
        feature_columns,
    )

    prepared_repeated_sets[experiment_name] = {
        "X_model": X_model,
        "categorical_columns": categorical_columns,
    }

    print(
        f"{experiment_name}: "
        f"{X_model.shape[1]} features"
    )

full_v6: 58 features
full_minus_color: 57 features


In [15]:
def normalize_title_for_shift(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .fillna("__MISSING__")
        .str.upper()
        .str.strip()
    )


def evaluate_holdout_seed(
    X_model: pd.DataFrame,
    categorical_columns: list[str],
    split_seed: int,
):
    price_bins = pd.qcut(
        y,
        q=10,
        labels=False,
        duplicates="drop",
    )

    train_idx_seed, valid_idx_seed = train_test_split(
        np.arange(len(y)),
        test_size=0.20,
        random_state=split_seed,
        stratify=price_bins,
    )

    X_train_fold = X_model.iloc[train_idx_seed].copy()
    X_valid_fold = X_model.iloc[valid_idx_seed].copy()

    y_train_fold = y.iloc[train_idx_seed].copy()
    y_valid_fold = y.iloc[valid_idx_seed].copy()

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    valid_pred = np.maximum(
        np.expm1(model.predict(X_valid_fold)),
        1,
    )

    title_train = normalize_title_for_shift(
        X_raw.iloc[train_idx_seed]["Полное название"]
    )

    title_valid = normalize_title_for_shift(
        X_raw.iloc[valid_idx_seed]["Полное название"]
    )

    unseen_mask = (
        ~title_valid.isin(set(title_train))
    ).to_numpy()

    low_price_threshold = y_train_fold.quantile(0.25)

    low_price_mask = (
        y_valid_fold.to_numpy() <= low_price_threshold
    )

    return {
        "best_iteration": model.get_best_iteration(),
        "overall_mape_pct": mape_percent(
            y_valid_fold,
            valid_pred,
        ),
        "unseen_title_mape_pct": mape_percent(
            y_valid_fold.to_numpy()[unseen_mask],
            valid_pred[unseen_mask],
        ),
        "low_price_mape_pct": mape_percent(
            y_valid_fold.to_numpy()[low_price_mask],
            valid_pred[low_price_mask],
        ),
    }

In [16]:
split_seeds = [42, 202, 2026]

repeated_holdout_rows = []

for split_seed in split_seeds:
    for experiment_name, data in prepared_repeated_sets.items():
        print(
            f"\n{'=' * 70}"
        )
        print(
            f"Seed={split_seed} | {experiment_name}"
        )
        print(
            f"{'=' * 70}"
        )

        result = evaluate_holdout_seed(
            X_model=data["X_model"],
            categorical_columns=data[
                "categorical_columns"
            ],
            split_seed=split_seed,
        )

        repeated_holdout_rows.append(
            {
                "split_seed": split_seed,
                "experiment": experiment_name,
                **result,
            }
        )

repeated_holdout_results = pd.DataFrame(
    repeated_holdout_rows
).sort_values(
    ["split_seed", "experiment"]
).reset_index(drop=True)

display(repeated_holdout_results)


Seed=42 | full_v6
0:	learn: 0.6538509	test: 0.6509994	best: 0.6509994 (0)	total: 76.5ms	remaining: 3m 49s
500:	learn: 0.1568077	test: 0.2042800	best: 0.2042780 (499)	total: 50.4s	remaining: 4m 11s
1000:	learn: 0.1200134	test: 0.1934439	best: 0.1934330 (994)	total: 2m 1s	remaining: 4m 2s
1500:	learn: 0.0982768	test: 0.1898236	best: 0.1897616 (1489)	total: 3m 6s	remaining: 3m 5s
2000:	learn: 0.0823696	test: 0.1882054	best: 0.1881731 (1994)	total: 4m 27s	remaining: 2m 13s
2500:	learn: 0.0693404	test: 0.1871944	best: 0.1871944 (2500)	total: 5m 50s	remaining: 1m 9s
2999:	learn: 0.0585626	test: 0.1864078	best: 0.1864078 (2999)	total: 7m 19s	remaining: 0us

bestTest = 0.1864077782
bestIteration = 2999


Seed=42 | full_minus_color
0:	learn: 0.6525543	test: 0.6505617	best: 0.6505617 (0)	total: 67ms	remaining: 3m 21s
500:	learn: 0.1551241	test: 0.2024758	best: 0.2024758 (500)	total: 1m 2s	remaining: 5m 13s
1000:	learn: 0.1189818	test: 0.1929087	best: 0.1928878 (999)	total: 2m 5s	remaining: 4m 1

,split_seed,experiment,best_iteration,overall_mape_pct,unseen_title_mape_pct,low_price_mape_pct
0,42,full_minus_color,2968,12.806429,16.547210,18.769882
1,42,full_v6,2999,12.908705,16.565257,18.724338
2,202,full_minus_color,2970,12.033115,15.442614,16.224179
3,202,full_v6,2981,12.249562,15.760950,16.101464
4,2026,full_minus_color,2999,12.548894,16.219305,17.146229
5,2026,full_v6,2762,12.709932,16.569550,17.081345


In [17]:
repeated_pivot = repeated_holdout_results.pivot(
    index="split_seed",
    columns="experiment",
    values="overall_mape_pct",
).reset_index()

repeated_pivot["v7_minus_v6"] = (
    repeated_pivot["full_minus_color"]
    - repeated_pivot["full_v6"]
)

display(repeated_pivot)

print(
    "Средняя разница v7 - v6:",
    f"{repeated_pivot['v7_minus_v6'].mean():.4f} п.п.",
)

print(
    "v7 лучше v6 на split:",
    (
        repeated_pivot["v7_minus_v6"] < 0
    ).sum(),
    "из",
    len(repeated_pivot),
)

experiment,split_seed,full_minus_color,full_v6,v7_minus_v6
0,42,12.806429,12.908705,-0.102276
1,202,12.033115,12.249562,-0.216447
2,2026,12.548894,12.709932,-0.161038


Средняя разница v7 - v6: -0.1599 п.п.
v7 лучше v6 на split: 3 из 3
